In [1]:
import pickle
import random
import numpy as np


def evaluate_rosso_policy_worst_case_expected_infinite(
    strategy_path,
    attacker_history_length=1,
    rollouts_num=1000,
    rollout_length=1050,
    protect_current_vertex=False,
):

    # =========================
    # LOAD STRATEGY
    # =========================

    with open(strategy_path, "rb") as f:
        P = pickle.load(f)

    P = np.array(P)

    # =========================
    # SAN FRANCISCO INSTANCE
    # =========================

    adj_matrix = [
        [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
        [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
        [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
        [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
        [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
        [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
        [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
        [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
        [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
        [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
        [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
        [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1],
    ]

    coverage = [
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    ]

    target_values = [1] * 12

    attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]

    num_nodes = len(adj_matrix)

    # =========================
    # HELPERS
    # =========================

    def sample_next(current):
        return random.choices(
            range(num_nodes),
            weights=P[current],
        )[0]

    def expected_capture_probability(path, target, tau):

        survive_prob = 1.0

        if protect_current_vertex:
            survive_prob *= (1.0 - coverage[path[0]][target])

        for i in range(len(path) - 1):

            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]

            if tau >= 0:
                survive_prob *= (1.0 - coverage[next_pos][target])
            else:
                break

        return 1.0 - survive_prob

    # =========================
    # HOW MANY ATTACK STARTS
    # =========================

    max_tau = max(attack_duration)
    usable_steps = rollout_length - max_tau - 10 # just to be safe

    observation_rewards = {}

    # =========================
    # MAIN LOOP
    # =========================

    for rollout_id in range(rollouts_num):

        if rollout_id % 100 == 0:
            print(f"Rollout {rollout_id}/{rollouts_num}")

        rollout = []

        current = random.randrange(num_nodes)
        rollout.append(current)

        for _ in range(rollout_length - 1):
            current = sample_next(current)
            rollout.append(current)

        # ----- evaluate every possible attack start -----

        for start in range(1, usable_steps + 1):

            if attacker_history_length == -1:
                observation = tuple(rollout[:start])
            else:
                observation = tuple(
                    rollout[
                        max(0, start - attacker_history_length):start
                    ]
                )

            attack_path = rollout[start - 1:]

            for target in range(num_nodes):

                if target_values[target] == 0:
                    continue

                capture_prob = expected_capture_probability(
                    attack_path,
                    target,
                    attack_duration[target],
                )

                reward = capture_prob * target_values[target]

                key = (observation, target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)

    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    for key, vals in observation_rewards.items():

        mean_reward = sum(vals) / len(vals)

        obs_target_rewards[key] = mean_reward
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():

        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]

    print("\nWorst case:", worst_pair)
    print("Worst-case value:", worst_value)
    print(f"\nWorst-case value in %: {worst_value * 100:.2f}")
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)

    return worst_value

In [2]:
strategy_path = "../results/local/demo_1/strategy_best_5_initializations.pkl"

evaluate_rosso_policy_worst_case_expected_infinite(
    strategy_path,
    attacker_history_length=1,
    rollouts_num=1000,
    rollout_length=1050,
    protect_current_vertex=True,
)

Rollout 0/1000
Rollout 100/1000
Rollout 200/1000
Rollout 300/1000
Rollout 400/1000
Rollout 500/1000
Rollout 600/1000
Rollout 700/1000
Rollout 800/1000
Rollout 900/1000

Worst case: ((6,), 8)
Worst-case value: 0.045046455788704455

Worst-case value in %: 4.50

Count: 95790
{((7,), 0): 0.4175393770276858, ((7,), 1): 0.14838037284203803, ((7,), 2): 0.2598631679806351, ((7,), 3): 0.09916118946359831, ((7,), 4): 0.07044999915950848, ((7,), 5): 0.13273042075005465, ((7,), 6): 0.13671435055220293, ((7,), 7): 1.0, ((7,), 8): 0.09347946679218007, ((7,), 9): 0.26999949570508835, ((7,), 10): 0.10149775588764309, ((7,), 11): 0.09655566575333255, ((2,), 0): 0.39413510211206143, ((2,), 1): 0.09835251151362165, ((2,), 2): 1.0, ((2,), 3): 0.13629711186003732, ((2,), 4): 0.06218027041905555, ((2,), 5): 0.1628019388536058, ((2,), 6): 0.13883480806154921, ((2,), 7): 0.15645098486781153, ((2,), 8): 0.06077043919599339, ((2,), 9): 0.1462733461337057, ((2,), 10): 0.11983565396028303, ((2,), 11): 0.064637404

0.045046455788704455